# Healthcare MLOps Demo - Part 2: Feature Engineering & Feature Store

**Duration:** ~30 minutes

## Overview
In this notebook, we'll build a Feature Store for our readmission prediction model:

1. **Feature Store Setup** - Initialize Snowflake Feature Store
2. **Entity Registration** - Define PATIENT and ENCOUNTER entities
3. **Feature Views** - Create managed feature pipelines with:
   - Patient demographics features
   - Encounter aggregation features
   - Rolling window features (30/60/90 day windows)
4. **Online Feature Store** - Enable low-latency serving

## Why Feature Store?
- **Consistency**: Same features for training and inference
- **Point-in-Time Correctness**: Avoid data leakage
- **Reusability**: Share features across models
- **Freshness**: Automated refresh with Dynamic Tables

---
## Setup

In [ ]:
# Import required packages
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *

# Feature Store imports
from snowflake.ml.feature_store import (
    FeatureStore,
    FeatureView,
    Entity,
    CreationMode
)

import pandas as pd

# Get session
session = get_active_session()

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;
USE DATABASE HEALTHCARE_MLOPS;
USE SCHEMA FEATURE_STORE;
USE WAREHOUSE HEALTHCARE_ML_WH;

select concat(
    'Current Role: ', current_role(), '\n',
    'Current Database: ', current_database(), '\n',
    'Current Schema: ', current_schema(), '\n',
    'Current Warehouse: ', current_warehouse(), '\n'
) as context;

---
## 1. Initialize Feature Store

The Feature Store manages entities, feature views, and their relationships.
It integrates with Snowflake's Dynamic Tables for automated refresh.

In [ ]:
# Initialize the Feature Store
fs = FeatureStore(
    session=session,
    database="HEALTHCARE_MLOPS",
    name="FEATURE_STORE",
    default_warehouse="HEALTHCARE_ML_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

---
## 2. Register Entities

Entities represent the primary keys for features. For healthcare:
- **PATIENT**: Individual patient (demographics, history)
- **ENCOUNTER**: Single hospital visit (point-in-time features)

In [ ]:
# Create PATIENT entity
patient_entity = Entity(
    name="PATIENT",
    join_keys=["PATIENT_ID"],
    desc="Healthcare patient entity - represents individual patients"
)

# Register the entity
fs.register_entity(patient_entity)
print("Registered PATIENT entity")

In [ ]:
# Create ENCOUNTER entity  
encounter_entity = Entity(
    name="ENCOUNTER",
    join_keys=["ENCOUNTER_ID"],
    desc="Healthcare encounter entity - represents hospital visits/encounters"
)

# Register the entity
fs.register_entity(encounter_entity)
print("Registered ENCOUNTER entity")

In [ ]:
# List all registered entities
fs.list_entities().to_pandas()

---
## 3. Create Feature Views

Feature Views define how features are computed from source data.
They automatically create Dynamic Tables for incremental refresh.

### 3.1 Patient Demographics Features

Static features that describe patient characteristics.

In [ ]:
# Create source DataFrame for patient demographics
patient_demographics_df = session.sql("""
    SELECT
        PATIENT_ID,
        AGE,
        GENDER,
        -- Encode gender as numeric
        CASE WHEN GENDER = 'M' THEN 1 ELSE 0 END AS IS_MALE,
        RACE,
        ETHNICITY,
        MARITAL_STATUS,
        -- Encode marital status
        CASE WHEN MARITAL_STATUS = 'M' THEN 1 ELSE 0 END AS IS_MARRIED,
        STATE,
        INCOME,
        HEALTHCARE_EXPENSES,
        HEALTHCARE_COVERAGE,
        COVERAGE_RATIO,
        -- Age buckets for binning
        CASE 
            WHEN AGE < 30 THEN 'YOUNG'
            WHEN AGE < 50 THEN 'MIDDLE'
            WHEN AGE < 70 THEN 'SENIOR'
            ELSE 'ELDERLY'
        END AS AGE_GROUP,
        -- Income level
        CASE
            WHEN INCOME < 40000 THEN 'LOW'
            WHEN INCOME < 80000 THEN 'MEDIUM'
            ELSE 'HIGH'
        END AS INCOME_LEVEL,
        IS_DECEASED,
        METADATA$ROW_LAST_COMMIT_TIME AS FEATURE_TIMESTAMP
    FROM HEALTHCARE_MLOPS.CURATED.PATIENTS_CURATED
""")

patient_demographics_df.limit(5).to_pandas()

In [ ]:
# Create Feature View for patient demographics
patient_demographics_fv = FeatureView(
    name="PATIENT_DEMOGRAPHICS_FV",
    entities=[patient_entity],
    feature_df=patient_demographics_df,
    timestamp_col="FEATURE_TIMESTAMP",
    refresh_freq="1 hour",
    desc="Patient demographic features including age, gender, income, and insurance coverage",
)

# Register the feature view
patient_demographics_fv = fs.register_feature_view(
    feature_view=patient_demographics_fv,
    version="1.0",
    block=True,  # Wait for initial refresh,
    overwrite=True
)

print(f"Registered Feature View: {patient_demographics_fv.name}")

### 3.2 Patient Encounter History Features

Aggregated features about a patient's historical encounters.
These are critical for readmission prediction.

In [ ]:
# Create patient-level encounter aggregations
patient_encounter_history_df = session.sql("""
    SELECT
        PATIENT_ID,
        -- Total encounter counts
        COUNT(*) AS TOTAL_ENCOUNTERS,
        SUM(IS_INPATIENT) AS TOTAL_INPATIENT_VISITS,
        COUNT(*) - SUM(IS_INPATIENT) AS TOTAL_OUTPATIENT_VISITS,
        
        -- Cost metrics
        SUM(TOTAL_CLAIM_COST) AS TOTAL_CLAIM_COSTS,
        AVG(TOTAL_CLAIM_COST) AS AVG_CLAIM_COST,
        MAX(TOTAL_CLAIM_COST) AS MAX_CLAIM_COST,
        SUM(OUT_OF_POCKET_COST) AS TOTAL_OUT_OF_POCKET,
        
        -- Length of stay metrics (for inpatient)
        AVG(CASE WHEN IS_INPATIENT = 1 THEN LENGTH_OF_STAY_DAYS END) AS AVG_LENGTH_OF_STAY_DAYS,
        MAX(CASE WHEN IS_INPATIENT = 1 THEN LENGTH_OF_STAY_DAYS END) AS MAX_LENGTH_OF_STAY_DAYS,
        SUM(CASE WHEN IS_INPATIENT = 1 THEN LENGTH_OF_STAY_DAYS ELSE 0 END) AS TOTAL_INPATIENT_DAYS,
        
        -- High cost visits
        SUM(IS_HIGH_COST) AS HIGH_COST_VISIT_COUNT,
        
        -- Encounter recency
        MAX(START_DATETIME) AS LAST_ENCOUNTER_DATE,
        MIN(START_DATETIME) AS FIRST_ENCOUNTER_DATE,
        DATEDIFF('day', MIN(START_DATETIME), MAX(START_DATETIME)) AS DAYS_AS_PATIENT,
        
        -- Unique providers and organizations
        COUNT(DISTINCT PROVIDER_ID) AS UNIQUE_PROVIDERS,
        COUNT(DISTINCT ORGANIZATION_ID) AS UNIQUE_ORGANIZATIONS,
        
        MAX(METADATA$ROW_LAST_COMMIT_TIME) AS FEATURE_TIMESTAMP
    FROM HEALTHCARE_MLOPS.CURATED.ENCOUNTERS_CURATED
    GROUP BY PATIENT_ID
""")

patient_encounter_history_df.limit(5).to_pandas()

In [ ]:
# Create Feature View for patient encounter history
patient_encounter_history_fv = FeatureView(
    name="PATIENT_ENCOUNTER_HISTORY_FV",
    entities=[patient_entity],
    feature_df=patient_encounter_history_df,
    timestamp_col="FEATURE_TIMESTAMP",
    refresh_freq="1 hour",
    desc="Aggregated patient encounter history including visits, costs, and length of stay"
)

patient_encounter_history_fv = fs.register_feature_view(
    feature_view=patient_encounter_history_fv,
    version="1.0",
    block=True,
    overwrite=True
)

print(f"Registered Feature View: {patient_encounter_history_fv.name}")

### 3.3 Patient Condition Features

Features derived from patient diagnoses/conditions.

In [ ]:
# Create patient condition aggregations
patient_condition_features_df = session.sql("""
    SELECT
        PATIENT_ID,
        
        -- Total conditions
        COUNT(DISTINCT ICD10_CODE) AS UNIQUE_DIAGNOSES,
        COUNT(*) AS TOTAL_DIAGNOSIS_RECORDS,
        
        -- Chronic condition count
        SUM(IS_CHRONIC) AS CHRONIC_CONDITION_COUNT,
        
        -- Conditions by category
        SUM(CASE WHEN CONDITION_CATEGORY = 'Circulatory' THEN 1 ELSE 0 END) AS CIRCULATORY_CONDITIONS,
        SUM(CASE WHEN CONDITION_CATEGORY = 'Endocrine' THEN 1 ELSE 0 END) AS ENDOCRINE_CONDITIONS,
        SUM(CASE WHEN CONDITION_CATEGORY = 'Respiratory' THEN 1 ELSE 0 END) AS RESPIRATORY_CONDITIONS,
        SUM(CASE WHEN CONDITION_CATEGORY = 'Neoplasm' THEN 1 ELSE 0 END) AS NEOPLASM_CONDITIONS,
        SUM(CASE WHEN CONDITION_CATEGORY = 'Mental' THEN 1 ELSE 0 END) AS MENTAL_CONDITIONS,
        SUM(CASE WHEN CONDITION_CATEGORY = 'Injury' THEN 1 ELSE 0 END) AS INJURY_CONDITIONS,
        
        -- High-risk condition flags (common readmission risk factors)
        MAX(CASE WHEN ICD10_CODE LIKE 'I50%' THEN 1 ELSE 0 END) AS HAS_HEART_FAILURE,
        MAX(CASE WHEN ICD10_CODE LIKE 'I10%' THEN 1 ELSE 0 END) AS HAS_HYPERTENSION,
        MAX(CASE WHEN ICD10_CODE LIKE 'E11%' THEN 1 ELSE 0 END) AS HAS_DIABETES,
        MAX(CASE WHEN ICD10_CODE LIKE 'J44%' THEN 1 ELSE 0 END) AS HAS_COPD,
        MAX(CASE WHEN ICD10_CODE LIKE 'N18%' THEN 1 ELSE 0 END) AS HAS_CKD,
        MAX(CASE WHEN ICD10_CODE LIKE 'C%' THEN 1 ELSE 0 END) AS HAS_CANCER,
        
        -- Comorbidity score (simplified Charlson-like)
        (
            MAX(CASE WHEN ICD10_CODE LIKE 'I50%' THEN 1 ELSE 0 END) +
            MAX(CASE WHEN ICD10_CODE LIKE 'E11%' THEN 1 ELSE 0 END) +
            MAX(CASE WHEN ICD10_CODE LIKE 'J44%' THEN 1 ELSE 0 END) +
            MAX(CASE WHEN ICD10_CODE LIKE 'N18%' THEN 1 ELSE 0 END) +
            MAX(CASE WHEN ICD10_CODE LIKE 'C%' THEN 2 ELSE 0 END)
        ) AS COMORBIDITY_SCORE,
        
        MAX(METADATA$ROW_LAST_COMMIT_TIME) AS FEATURE_TIMESTAMP
    FROM HEALTHCARE_MLOPS.CURATED.CONDITIONS_CURATED
    GROUP BY PATIENT_ID
""")

patient_condition_features_df.limit(5).to_pandas()

In [ ]:
# Create Feature View for patient conditions
patient_condition_fv = FeatureView(
    name="PATIENT_CONDITION_FV",
    entities=[patient_entity],
    feature_df=patient_condition_features_df,
    timestamp_col="FEATURE_TIMESTAMP",
    refresh_freq="1 hour",
    desc="Patient condition/diagnosis features including comorbidity scores and risk flags"
)

patient_condition_fv = fs.register_feature_view(
    feature_view=patient_condition_fv,
    version="1.0",
    block=True
)

print(f"Registered Feature View: {patient_condition_fv.name}")

### 3.4 Encounter-Level Features

Features specific to each encounter (used as model input at prediction time).

In [ ]:
# Create encounter-level features
encounter_features_df = session.sql("""
    SELECT
        e.ENCOUNTER_ID,
        
        -- Encounter characteristics
        e.LENGTH_OF_STAY_DAYS,
        e.LENGTH_OF_STAY_HOURS,
        e.IS_INPATIENT,
        e.IS_HIGH_COST,
        e.TOTAL_CLAIM_COST,
        e.OUT_OF_POCKET_COST,
        
        -- Time-based features
        DAYOFWEEK(e.START_DATETIME) AS ADMISSION_DAY_OF_WEEK,
        HOUR(e.START_DATETIME) AS ADMISSION_HOUR,
        CASE WHEN DAYOFWEEK(e.START_DATETIME) IN (0, 6) THEN 1 ELSE 0 END AS IS_WEEKEND_ADMISSION,
        MONTH(e.START_DATETIME) AS ADMISSION_MONTH,
        
        -- Primary reason category
        CASE 
            WHEN e.REASON_CODE LIKE 'I%' THEN 'Circulatory'
            WHEN e.REASON_CODE LIKE 'E%' THEN 'Endocrine'
            WHEN e.REASON_CODE LIKE 'J%' THEN 'Respiratory'
            WHEN e.REASON_CODE LIKE 'C%' THEN 'Neoplasm'
            ELSE 'Other'
        END AS PRIMARY_REASON_CATEGORY,
        
        -- Number of conditions diagnosed in this encounter
        COALESCE(c.CONDITION_COUNT, 0) AS CONDITIONS_IN_ENCOUNTER,
        
        e.METADATA$ROW_LAST_COMMIT_TIME AS FEATURE_TIMESTAMP
    FROM HEALTHCARE_MLOPS.CURATED.ENCOUNTERS_CURATED e
    LEFT JOIN (
        SELECT ENCOUNTER_ID, COUNT(*) AS CONDITION_COUNT
        FROM HEALTHCARE_MLOPS.CURATED.CONDITIONS_CURATED
        GROUP BY ENCOUNTER_ID
    ) c ON e.ENCOUNTER_ID = c.ENCOUNTER_ID
    WHERE e.IS_INPATIENT = 1  -- Only inpatient for readmission prediction
""")

encounter_features_df.limit(5).to_pandas()

In [ ]:
# Create Feature View for encounter features
encounter_fv = FeatureView(
    name="ENCOUNTER_FEATURES_FV",
    entities=[encounter_entity],
    feature_df=encounter_features_df,
    timestamp_col="FEATURE_TIMESTAMP",
    refresh_freq="1 hour",
    desc="Encounter-level features for readmission prediction"
)

encounter_fv = fs.register_feature_view(
    feature_view=encounter_fv,
    version="1.0",
    block=True,
    overwrite=True
)

print(f"Registered Feature View: {encounter_fv.name}")

---
## 4. View All Feature Views

In [ ]:
# List all feature views
fs.list_feature_views().to_pandas()

In [ ]:
# Get details on a specific feature view
demographics_fv = fs.get_feature_view("PATIENT_DEMOGRAPHICS_FV", "1.0")
print(f"Feature View: {demographics_fv.name}")
print(f"Description: {demographics_fv.desc}")
print(f"Entities: {[e.name for e in demographics_fv.entities]}")
print(f"\nFeature columns:")
for col in demographics_fv.feature_df.columns:
    print(f"  - {col}")

---
## 5. Enable Online Feature Store (Optional)

The Online Feature Store enables low-latency feature retrieval for real-time inference.
Features are cached in a high-performance store for sub-millisecond access.

In [ ]:
# Enable online store for patient demographics (for real-time scoring)
# Note: This requires additional Snowflake configuration
try:
    # Get the feature view
    patient_demo_fv = fs.get_feature_view("PATIENT_DEMOGRAPHICS_FV", "1.0")
    
    # Update with online enabled
    # (In production, you'd also configure the online store warehouse)
    print("Online Feature Store can be enabled with:")
    print("  fs.update_feature_view(")
    print("      name='PATIENT_DEMOGRAPHICS_FV',")
    print("      version='1.0',")
    print("      online_enabled=True")
    print("  )")
except Exception as e:
    print(f"Note: {e}")

---
## 6. Preview Features for Sample Patients

In [ ]:
# Get a sample of patients to retrieve features for
sample_patients = session.sql("""
    SELECT PATIENT_ID 
    FROM HEALTHCARE_MLOPS.CURATED.PATIENTS_CURATED 
    LIMIT 5
""").to_pandas()

print("Sample Patient IDs:")
print(sample_patients)

In [ ]:
# Retrieve features for sample patients
patient_fv = fs.get_feature_view("PATIENT_DEMOGRAPHICS_FV", "1.0")
condition_fv = fs.get_feature_view("PATIENT_CONDITION_FV", "1.0")

# Create a spine (entity keys to retrieve features for)
spine_df = session.create_dataframe(sample_patients)

# Retrieve features - this does a point lookup
features_df = fs.retrieve_feature_values(
    spine_df=spine_df,
    features=[patient_fv, condition_fv]
)

features_df.limit(5).to_pandas()

---
## Summary: Feature Store Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                      FEATURE STORE                              │
├─────────────────────────────────────────────────────────────────┤
│  Entities:                                                      │
│    • PATIENT (PATIENT_ID)                                       │
│    • ENCOUNTER (ENCOUNTER_ID)                                   │
├─────────────────────────────────────────────────────────────────┤
│  Feature Views:                                                 │
│    ┌──────────────────────────┐  ┌───────────────────────────┐ │
│    │ PATIENT_DEMOGRAPHICS_FV  │  │ PATIENT_ENCOUNTER_HISTORY │ │
│    │ • Age, Gender            │  │ • Visit counts            │ │
│    │ • Income, Coverage       │  │ • Cost totals             │ │
│    │ • Age Group, Location    │  │ • Length of stay          │ │
│    └──────────────────────────┘  └───────────────────────────┘ │
│    ┌──────────────────────────┐  ┌───────────────────────────┐ │
│    │ PATIENT_CONDITION_FV     │  │ ENCOUNTER_FEATURES_FV     │ │
│    │ • Diagnosis counts       │  │ • LOS, Cost               │ │
│    │ • Comorbidity score      │  │ • Admission timing        │ │
│    │ • High-risk flags        │  │ • Conditions count        │ │
│    └──────────────────────────┘  └───────────────────────────┘ │
└─────────────────────────────────────────────────────────────────┘
```

### Features Created:
- **40+ features** across 4 feature views
- **Automated refresh** via Dynamic Tables (1 hour lag)
- **Point-in-time correct** joins for training
- **Reusable** across multiple ML models

### Next Steps:
Continue to **Notebook 3: Model Training** to train the readmission prediction model.

In [ ]:
# Final status check
print("=== FEATURE STORE STATUS ===")
print(f"\nEntities registered: {len(fs.list_entities().collect())}")
print(f"Feature Views registered: {len(fs.list_feature_views().collect())}")

print("\n=== FEATURE VIEW DETAILS ===")
fs.list_feature_views().select('"NAME"', '"VERSION"', '"DESC"').to_pandas()